In [1]:
# imports
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

In [2]:
# loading training dataset
df = pd.read_csv("train_mlp_extended_all.csv")

print(df.shape)

df.head()

(4527, 4)


,quality_score,best_similarity,margin,label
0,27.075550,0.574058,0.413278,1
1,15.175558,0.443810,0.227276,1
2,20.051987,0.505812,0.315841,1
3,16.646429,0.373729,0.075268,1
4,19.913544,0.537300,0.352941,1


In [3]:
# features and labels
X = df[
    [
        "quality_score",
        "best_similarity",
        "margin",
    ]
]

y = df["label"]

print(X.shape)
print(y.shape)

(4527, 3)
(4527,)


In [4]:
# train validation split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(X_train.shape)
print(X_val.shape)

(3621, 3)
(906, 3)


In [5]:
# scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

joblib.dump(
    scaler,
    "mlp_scaler_exp1.pkl",
)

print("Scaler Saved")

Scaler Saved


In [6]:
# tensor conversion
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32,
)

X_val_tensor = torch.tensor(
    X_val_scaled,
    dtype=torch.float32,
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.long,
)

y_val_tensor = torch.tensor(
    y_val.values,
    dtype=torch.long,
)

In [7]:
# MLP architecture
class MLPExperiment1(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def forward(self, x):

        return self.network(x)

In [8]:
# model setup
model = MLPExperiment1()

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4,
)

print(model)

MLPExperiment1(
  (network): Sequential(
    (0): Linear(in_features=3, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=2, bias=True)
  )
)


In [9]:
# training the model
epochs = 300

best_val_accuracy = 0

for epoch in range(epochs):

    model.train()

    outputs = model(X_train_tensor)

    loss = criterion(
        outputs,
        y_train_tensor,
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    model.eval()

    with torch.no_grad():

        val_outputs = model(X_val_tensor)

        predictions = torch.argmax(
            val_outputs,
            dim=1,
        )

        val_accuracy = accuracy_score(
            y_val_tensor.numpy(),
            predictions.numpy(),
        )

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            "mlp_exp1_model.pth",
        )

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:3d} | "
            f"Loss {loss.item():.4f} | "
            f"Val Acc {val_accuracy:.4f}"
        )

print()

print(
    "Best Validation Accuracy:",
    best_val_accuracy,
)

Epoch   0 | Loss 0.6454 | Val Acc 0.5188
Epoch  10 | Loss 0.6345 | Val Acc 0.5596
Epoch  20 | Loss 0.6236 | Val Acc 0.6038
Epoch  30 | Loss 0.6119 | Val Acc 0.6336
Epoch  40 | Loss 0.5992 | Val Acc 0.6667
Epoch  50 | Loss 0.5854 | Val Acc 0.6998
Epoch  60 | Loss 0.5707 | Val Acc 0.7395
Epoch  70 | Loss 0.5550 | Val Acc 0.7770
Epoch  80 | Loss 0.5384 | Val Acc 0.8146
Epoch  90 | Loss 0.5210 | Val Acc 0.8819
Epoch 100 | Loss 0.5028 | Val Acc 0.9294
Epoch 110 | Loss 0.4839 | Val Acc 0.9503
Epoch 120 | Loss 0.4645 | Val Acc 0.9669
Epoch 130 | Loss 0.4451 | Val Acc 0.9680
Epoch 140 | Loss 0.4260 | Val Acc 0.9691
Epoch 150 | Loss 0.4075 | Val Acc 0.9702
Epoch 160 | Loss 0.3894 | Val Acc 0.9702
Epoch 170 | Loss 0.3719 | Val Acc 0.9713
Epoch 180 | Loss 0.3548 | Val Acc 0.9713
Epoch 190 | Loss 0.3380 | Val Acc 0.9713
Epoch 200 | Loss 0.3213 | Val Acc 0.9724
Epoch 210 | Loss 0.3045 | Val Acc 0.9735
Epoch 220 | Loss 0.2874 | Val Acc 0.9735
Epoch 230 | Loss 0.2705 | Val Acc 0.9746
Epoch 240 | Loss

In [10]:
# loading the best model
best_model = MLPExperiment1()

best_model.load_state_dict(
    torch.load(
        "mlp_exp1_model.pth",
        map_location=torch.device("cpu"),
    )
)

best_model.eval()

print("Best Model Loaded")

Best Model Loaded


In [11]:
# validation metrics
with torch.no_grad():

    outputs = best_model(X_val_tensor)

    predictions = torch.argmax(
        outputs,
        dim=1,
    )

y_pred = predictions.numpy()

y_true = y_val_tensor.numpy()

In [12]:
# final metrics (acfuracy, precision, F1, Recall)
accuracy = accuracy_score(
    y_true,
    y_pred,
)

precision = precision_score(
    y_true,
    y_pred,
)

recall = recall_score(
    y_true,
    y_pred,
)

f1 = f1_score(
    y_true,
    y_pred,
)

print("Accuracy :", accuracy)

print("Precision:", precision)

print("Recall   :", recall)

print("F1 Score :", f1)

Accuracy : 0.977924944812362
Precision: 0.9823788546255506
Recall   : 0.9737991266375546
F1 Score : 0.9780701754385965


In [13]:
# confusion matrix
cm = confusion_matrix(
    y_true,
    y_pred,
)

print(cm)

print()

print(
    classification_report(
        y_true,
        y_pred,
    )
)

[[440   8]
 [ 12 446]]

              precision    recall  f1-score   support

           0       0.97      0.98      0.98       448
           1       0.98      0.97      0.98       458

    accuracy                           0.98       906
   macro avg       0.98      0.98      0.98       906
weighted avg       0.98      0.98      0.98       906



In [14]:
# TRR TAR FRR FAR
TN, FP, FN, TP = cm.ravel()

TAR = TP / (TP + FN)

FRR = FN / (TP + FN)

TRR = TN / (TN + FP)

FAR = FP / (TN + FP)

print()

print("TAR:", TAR)

print("FRR:", FRR)

print("TRR:", TRR)

print("FAR:", FAR)


TAR: 0.9737991266375546
FRR: 0.026200873362445413
TRR: 0.9821428571428571
FAR: 0.017857142857142856
